In [ ]:
import os
import logging
import chromadb
import re
import pathlib
from chromadb.config import Settings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

# 1. Configure Production Logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

In [ ]:
# FIX1
def _extract_field(text: str, field_name: str) -> str | None:
    """
    Case-insensitive field extractor with leading-whitespace tolerance.
    Logs a DEBUG entry when found, returns None silently when absent
    (callers are responsible for logging fallback warnings).
    """
    pattern = re.compile(
        rf"^\s*{re.escape(field_name)}\s*:\s*(.+)$",
        re.IGNORECASE | re.MULTILINE
    )
    match = pattern.search(text)
    if match:
        value = match.group(1).strip()
        return value if value else None
    return None

class ThreatIntelDB:
    def __init__(self, db_path: str = "./brain"):
        """Initializes the persistent vector database."""
        try:
            self.client = chromadb.PersistentClient(path=db_path)
            embedding_fn = SentenceTransformerEmbeddingFunction(
                model_name="all-MiniLM-L6-v2"
            )

            self.collection = self.client.get_or_create_collection(
                name="mitre_threat_intel",
                embedding_function=embedding_fn,  
                metadata={"hnsw:space": "cosine"} 
            )
            # Initialize the splitter for large documents (CISA reports, etc.)
            self.splitter = RecursiveCharacterTextSplitter(
                chunk_size=1000,
                chunk_overlap=200,
                separators=["\n\n", "\n", " ", ""]
            )
            logger.info(f"Successfully connected to ChromaDB at {db_path}")
        except Exception as e:
            logger.error(f"Failed to initialize ChromaDB: {e}")
            raise
    # PRODUCTION VERSION — delete old chunks for this file before upserting new ones
    def _delete_existing_chunks(self, filename: str):
        """Removes all previously ingested chunks for a given source file."""
        try:
            existing = self.collection.get(where={"source": filename})
            if existing and existing["ids"]:
                self.collection.delete(ids=existing["ids"])
                logger.info(f"[{filename}] Deleted {len(existing['ids'])} stale chunks "
                            f"before re-ingestion.")
        except Exception as e:
            logger.warning(f"[{filename}] Could not clean up old chunks: {e}")

    def process_directory(self, directory_path: str):
        """Iterates through a directory, chunks text, and prepares for ingestion."""
        dir_path = pathlib.Path(directory_path)
        if not dir_path.exists() or not dir_path.is_dir():
            logger.error(f"Path '{directory_path}' does not exist or is not a directory.")
            return
            
        txt_files = sorted(dir_path.glob("*.txt"))  # FIX5: deterministic order, no hidden files

        if not txt_files:
            logger.warning(f"No .txt files found in '{directory_path}'. Nothing to ingest.")
            return

        logger.info(f"Found {len(txt_files)} .txt files to process in '{directory_path}'.")

        for filepath in txt_files:
            filename = filepath.name
            if filename.endswith(".txt"):          # ← gate check FIRST
                filepath = os.path.join(dir_path, filename)
                try:                               # ← try sits INSIDE the if
                    with open(filepath, 'r', encoding='utf-8') as f:
                        raw_text = f.read().strip()
                        if not raw_text:  # FIX2
                            logger.warning(f"[{filename}] File is empty — skipping ingestion.")
                            continue  # move to next file in the loop

                        if len(raw_text) < 50:  # arbitrary minimum — a real threat report is never 10 chars
                            logger.warning(f"[{filename}] File suspiciously short ({len(raw_text)} chars) "
                                           f"— ingesting anyway but verify content.")

                    # In process_directory — warn on every fallback, don't silently accept "unknown"
                    technique_id = (
                        _extract_field(raw_text, "TECHNIQUE_ID") or
                        _extract_field(raw_text, "VULNERABILITY_ID")
                    )
                    if not technique_id:
                        technique_id = filename.replace(".txt", "")
                        logger.warning(f"[{filename}] No TECHNIQUE_ID or VULNERABILITY_ID found — "
                                       f"falling back to filename: '{technique_id}'")

                    tactic = _extract_field(raw_text, "TACTIC")
                    if not tactic:
                        tactic = "unknown"
                        logger.warning(f"[{filename}] No TACTIC field found — defaulting to 'unknown'")

                    date_added = _extract_field(raw_text, "DATE_ADDED")
                    if not date_added:
                        date_added = "unknown"
                        logger.warning(f"[{filename}] No DATE_ADDED field found — defaulting to 'unknown'")

                    chunks = self.splitter.create_documents(
                        texts=[raw_text],
                        metadatas=[{
                            "source":       filename,
                            "type":         "threat_intel_report",
                            "technique_id": technique_id,
                            "tactic":       tactic,
                            "date_added":   date_added
                        }]
                    )

                    documents = [c.page_content for c in chunks]
                    metadatas = [c.metadata for c in chunks]
                    ids       = [f"{filename}_{i}" for i in range(len(chunks))]

                    # FIX3: In process_directory, before self.collection.upsert(...)
                    self._delete_existing_chunks(filename)
                    self.collection.upsert(documents=documents, metadatas=metadatas, ids=ids)
                except Exception as e:
                    logger.error(f"Error processing {filename}: {e}")  # ← filename, not filepath
    
    # FIX4
    def semantic_search(self, query: str, n_results: int = 3):
        """
        Returns query results dict on success, empty result structure on no-match,
        or None on hard failure. Caller can distinguish all three cases.
        """
        if not query or not query.strip():
            logger.warning("semantic_search called with empty query — returning None.")
            return None

        if n_results < 1:
            logger.error(f"n_results must be >= 1, got {n_results}.")
            return None

        collection_count = self.collection.count()
        if collection_count == 0:
            logger.warning("semantic_search called on empty collection. "
                       "Run process_directory() first.")
            return None

        # Clamp n_results to what actually exists — prevents ChromaDB crash
        n_results = min(n_results, collection_count)

        try:
            results = self.collection.query(
                query_texts=[query.strip()],
                n_results=n_results
            )
            logger.info(f"Search returned {len(results['documents'][0])} results "
                    f"for query: '{query[:80]}'")
            return results
        except Exception as e:
            logger.error(f"Search failed for query '{query[:80]}': {e}", exc_info=True)
            return None

# --- Execution Block ---
if __name__ == "__main__":
    db = ThreatIntelDB()

    intel_dir = "./threat_reports"

    if not os.path.exists(intel_dir):
        os.makedirs(intel_dir)
        with open(f"{intel_dir}/sample_mitre.txt", "w") as f:
            f.write(
                "TECHNIQUE_ID: T1566\n"
                "TACTIC: Initial Access\n"
                "DATE_ADDED: 2024-01-15\n\n"
                "T1566 - Phishing\n\n"
                "Adversaries may send phishing messages to gain access to victim systems. "
                "This technique involves malicious links or attachments delivered via email. "
                "Defenders should monitor for suspicious email attachments and outbound connections "
                "to newly registered domains following email delivery events.\n"
        )


    db.process_directory(intel_dir)

    question = "How do hackers trick employees into clicking bad links to get inside the network?"
    search_results = db.semantic_search(question)

    if search_results and search_results['documents'][0]:
        logger.info("--- RAG RETRIEVAL RESULTS ---")
        for i in range(len(search_results['documents'][0])):
            doc  = search_results['documents'][0][i]
            meta = search_results['metadatas'][0][i]
            logger.info(f"Source: {meta['source']}")
            logger.info(f"Technique ID: {meta.get('technique_id', 'N/A')}")
            logger.info(f"Tactic: {meta.get('tactic', 'N/A')}")
            logger.info(f"Content: {doc[:300]}...")
    else:
        logger.warning("No results found. Check if documents were ingested correctly.")

In [ ]:
# ─── Cell: Imports + Hot-Reload ────────────────────────────────────────────
# sys.path ensures Python finds ingest.py and threat_analyzer.py in the same folder
# importlib.reload() guarantees the latest saved version is used on every
# re-run — without this, Jupyter caches the old module even after editing.
 
import os
import sys
import importlib
 
sys.path.append(os.getcwd())
 
import ingest
import threat_analyzer
 
importlib.reload(ingest)
importlib.reload(threat_analyzer)
 
from ingest      import ThreatIntelDB
from threat_analyzer  import ThreatAnalyzer
 
print("✅ Imports successful.")
print(f"   ingest    → {ingest.__file__}")
print(f"   generation → {threat_analyzer.__file__}")

In [ ]:

# ─── Cell: Initialize Both Systems + Ingest ────────────────────────────────
# ThreatIntelDB connects to (or creates) the persistent ChromaDB at ./brain
# process_directory() returns a stats dict — we validate it, not ignore it.
 
db       = ThreatIntelDB()
stats    = db.process_directory("./threat_reports")
analyzer = ThreatAnalyzer()
 
print("\n📦 Ingestion Summary:")
print(f"   processed    : {stats['processed']}")
print(f"   skipped      : {stats['skipped']}")
print(f"   failed       : {stats['failed']}")
print(f"   total_chunks : {stats['total_chunks']}")
 
# Sanity check — collection must be non-empty before querying
collection_count = db.collection.count()
print(f"\n🗄️  ChromaDB chunk count: {collection_count}")
 
assert collection_count > 0, (
    "❌ Collection is empty after ingestion. "
    "Check ./threat_reports has valid .txt files."
)
print("✅ Collection non-empty — safe to query.")

In [ ]:
# ─── Cell: End-to-End Pipeline Validation ──────────────────────────────────
# This cell exercises every layer of the pipeline in sequence and validates
# each one explicitly — not just "did it run" but "did it produce correct output".
 
question = "How do adversaries use phishing for initial access?"
 
# ── Layer 1: Semantic Search ──────────────────────────────────────────────────
search_results = db.semantic_search(question, n_results=3)
 
print("─" * 60)
print("🔍 LAYER 1: Semantic Search")
 
assert search_results is not None, "❌ semantic_search() returned None"
 
if search_results["error"]:
    print(f"❌ Search error: {search_results['error']}")
else:
    retrieved_docs = search_results["documents"][0]
    retrieved_meta = search_results["metadatas"][0]
    print(f"✅ Retrieved {len(retrieved_docs)} chunk(s). No upstream error.")
    for i, (doc, meta) in enumerate(zip(retrieved_docs, retrieved_meta)):
        print(f"\n   Chunk {i+1}:")
        print(f"     source       : {meta.get('source', 'N/A')}")
        print(f"     technique_id : {meta.get('technique_id', 'N/A')}")
        print(f"     tactic       : {meta.get('tactic', 'N/A')}")
        print(f"     preview      : {doc[:120].strip()}...")
 
# ── Layer 2: LLM Generation ───────────────────────────────────────────────────
result = analyzer.generate_answer(question, search_results)
 
print("\n" + "─" * 60)
print("🤖 LAYER 2: LLM Generation")
print(f"   success : {result.success}")
 
if not result.success:
    print(f"❌ Generation failed: {result.error}")
else:
    print(f"✅ Answer generated ({len(result.answer)} chars).")
    print(f"\n📝 ANSWER:\n{result.answer}")
 
    print("\n📎 SOURCE CITATIONS:")
    if result.source_citations:
        for c in result.source_citations:
            print(
                f"   - technique : {c.get('technique_id', 'N/A')}"
                f"  |  tactic : {c.get('tactic', 'N/A')}"
                f"  |  file : {c.get('source', 'N/A')}"
            )
    else:
        print("   ⚠️  No citations returned — check metadata extraction in ingest.py")
 
# ── Final Gate ────────────────────────────────────────────────────────────────
print("\n" + "─" * 60)
 
pipeline_ok = (
    search_results["error"] is None
    and len(search_results["documents"][0]) > 0
    and result.success
    and len(result.answer) > 0
    and len(result.source_citations) > 0
)
 
if pipeline_ok:
    print("✅ ALL LAYERS PASSED — pipeline is validated. Ready to build app.py.")
else:
    print("❌ PIPELINE INCOMPLETE — fix the failing layer above before building Gradio UI.")